# Parallel Tempering

Parallel tempering (a.k.a. replica exchange) runs several chains in parallel at *different inverse temperatures* $\beta$. The hot chains see a flattened density $\pi(x)^\beta$ that washes out energy barriers, while the cold chain ($\beta = 1$) targets the true posterior. Periodic swap proposals between adjacent temperatures let the cold chain inherit cross-mode jumps from its hotter neighbours.

This notebook shows how PT crosses a barrier that vanilla random-walk MH cannot.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import torch
from torch.distributions.multivariate_normal import MultivariateNormal
import matplotlib.pyplot as plt

from mcmc_samplers import MetropolisHastings, ParallelTempering, SampleVisualizer, diagnostics

## Bimodal target

Two well-separated isotropic Gaussians: a vanilla random walk cannot easily move between modes once it has localized in one.

In [ ]:
class BimodalGaussian:
    def __init__(self):
        self.m1 = MultivariateNormal(torch.tensor([-4.0, 0.0]), 0.4 * torch.eye(2))
        self.m2 = MultivariateNormal(torch.tensor([4.0, 0.0]), 0.4 * torch.eye(2))
    def __call__(self, x):
        lp = torch.stack([self.m1.log_prob(x), self.m2.log_prob(x)])
        return torch.logsumexp(lp, dim=0) - torch.log(torch.tensor(2.0))

target = BimodalGaussian()
x0 = torch.tensor([-4.0, 0.0])
N = 5000

In [ ]:
# Vanilla random-walk MH (will likely stay near the starting mode)
mh = MetropolisHastings(target=target, x0=x0, cov=torch.eye(2) * 0.5, generator=torch.Generator().manual_seed(0))
mh_samples, _ = mh(N)
print(f'MH acceptance: {100*mh.acceptance_ratio:.1f}%')

In [ ]:
# Parallel tempering: 4 replicas, geometric temperature ladder
def factory(tempered_target, x0_replica):
    return MetropolisHastings(
        target=tempered_target, x0=x0_replica, cov=torch.eye(2) * 0.5,
        generator=torch.Generator().manual_seed(42),
    )

pt = ParallelTempering(
    target=target,
    x0=x0,
    sampler_factory=factory,
    betas=(1.0, 0.3, 0.1, 0.03),
    generator=torch.Generator().manual_seed(0),
)
pt_samples, _ = pt(N)
print(f'PT swap acceptance per pair: {pt.swap_acceptance_ratios}')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
axs[0].scatter(mh_samples[:, 0], mh_samples[:, 1], s=2, alpha=0.4)
axs[0].set_title('Random-walk MH (single mode)')
axs[1].scatter(pt_samples[:, 0], pt_samples[:, 1], s=2, alpha=0.4)
axs[1].set_title('Parallel tempering (both modes)')
for ax in axs:
    ax.set_xlabel('$x_1$')
axs[0].set_ylabel('$x_2$')
plt.show()